In [1]:
import pandas as pd 
import numpy as np
import os
from pathlib import Path
from typing import Optional,Tuple, List

## 1.0 Import dataset

In [2]:
# import train data 
df = pd.read_csv("/home/sjoon/projects/brain_connectivity_classifier/data/raw/PIOP2_restingstate.csv")
df.shape

(224, 26797)

## 2. Data Preparation

In [3]:
# Extract connection columns
connection_columns = [col for col in df.columns if '~' in str(col)]

# Extract actual regions from connection columns
def extract_regions(connection_columns):
    unique_regions = []
    seen = set()
    
    for col in connection_columns:
        region_a, region_b = col.split('~', 1)
        for region in [region_a, region_b]:
            if region not in seen:
                seen.add(region)
                unique_regions.append(region)
    
    region_to_idx = {region: idx for idx, region in enumerate(unique_regions)}
    n_regions = len(unique_regions)
    
    return unique_regions, region_to_idx, n_regions

# Extract from your actual data
region_list, region_to_idx, n_regions = extract_regions(connection_columns)

# Print results
print(f"Found {n_regions} regions")
print(f"Sample regions: {region_list[:3]}")
print(f"Connection columns: {len(connection_columns)}")

Found 232 regions
Sample regions: ['LH_VisCent_ExStr_2', 'LH_VisCent_ExStr_1', 'LH_VisCent_Striate_1']
Connection columns: 26796


In [4]:
def reconstruct_matrices_from_dataframe(df, connection_columns, region_to_idx, n_regions):
    n_subjects = df.shape[0]
    matrices = np.zeros((n_subjects, n_regions, n_regions))
    
    values = df[connection_columns].values
    
    for subj_idx in range(n_subjects):
        matrix = matrices[subj_idx]
        for col_idx, col in enumerate(connection_columns):
            region_a, region_b = col.split('~', 1)
            idx_a = region_to_idx[region_a]
            idx_b = region_to_idx[region_b]
            
            # Place value symmetrically
            value = values[subj_idx, col_idx]
            matrix[idx_a, idx_b] = value
            matrix[idx_b, idx_a] = value
        
        # Self-correlations
        np.fill_diagonal(matrix, 1.0)
    
    return matrices 

df_mat = reconstruct_matrices_from_dataframe(df, connection_columns, region_to_idx, n_regions)
df_mat.shape

(224, 232, 232)

In [5]:
df_mat[0].shape

(232, 232)

## 3. Diagonal Imputation

In [6]:
def reconstruct_matrices_from_dataframe(df, connection_columns, region_to_idx, n_regions):
    n_subjects = df.shape[0]
    matrices = np.zeros((n_subjects, n_regions, n_regions))
    
    values = df[connection_columns].values
    
    for subj_idx in range(n_subjects):
        matrix = matrices[subj_idx]
        for col_idx, col in enumerate(connection_columns):
            region_a, region_b = col.split('~', 1)
            idx_a = region_to_idx[region_a]
            idx_b = region_to_idx[region_b]
            
            # Place value symmetrically
            value = values[subj_idx, col_idx]
            matrix[idx_a, idx_b] = value
            matrix[idx_b, idx_a] = value
        
        # Self-correlations
        np.fill_diagonal(matrix, 1.0)
    
    return matrices 

df_mat = reconstruct_matrices_from_dataframe(df, connection_columns, region_to_idx, n_regions)
df_mat.shape

(224, 232, 232)

In [7]:
# Extract diagonal and flatten
diagonal_values = np.diagonal(df_mat[0])

# first 30 diagonal values of subject 1
diagonal_values[0:30]

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])

In [8]:
# first 5 rows and columns of subject 0
df_mat[0][0:5, 0:5]

array([[1.        , 0.44923772, 0.52421013, 0.27670956, 0.32092815],
       [0.44923772, 1.        , 0.73007062, 0.49777039, 0.53532657],
       [0.52421013, 0.73007062, 1.        , 0.56103755, 0.32186132],
       [0.27670956, 0.49777039, 0.56103755, 1.        , 0.59854227],
       [0.32092815, 0.53532657, 0.32186132, 0.59854227, 1.        ]])

### 3.1 Impute diagonal with precision of off diagonal values 

In [9]:
## Precision matrix
def precision_full(
    matrices: np.ndarray,
    alpha: float = 0.1,
) -> np.ndarray:
    """
    Replace correlation matrices with precision matrices (full matrix, not just diagonal).
    
    Args:
        matrices: (n_subjects, n_regions, n_regions) correlation matrices
        alpha: Regularization strength for stability
    
    Returns:
        precision_matrices: (n_subjects, n_regions, n_regions) precision matrices
    """
    n_subjects, n_regions, _ = matrices.shape
    precision_matrices = np.zeros_like(matrices)
    
    for s in range(n_subjects):
        corr = matrices[s].copy()
        
        # Ensure valid correlation matrix
        np.fill_diagonal(corr, 1.0)
        corr = (corr + corr.T) / 2.0
        
        # Tikhonov regularization
        regularized = corr + alpha * np.eye(n_regions)
        
        # Invert to get precision matrix
        precision_matrices[s] = np.linalg.inv(regularized)
    
    return precision_matrices

df_precision = precision_full(df_mat, alpha=0.001)

print(f'5 rows x 5 col of subject 0 \n {df_precision[0][0:5, 0:5]} ')
print(f' \n 5 rows x 5 col of subject 1 \n {df_precision[1][0:5, 0:5]} ')

# Extract diagonal from first subject
diagonal_values_1 = np.diagonal(df_precision[0])
diagonal_values_2 = np.diagonal(df_precision[1])

# First 30 diagonal values of subject 0
print(f'First 30 diagonal values of subject 0 \n {diagonal_values_1[0:30]}')
print('')
print(f'First 30 diagonal values of subject 1 \n {diagonal_values_2[0:30]}')

5 rows x 5 col of subject 0 
 [[725.70683943 -14.31363221 -17.14047155 -24.84658715 -36.96701965]
 [-14.31363221 675.0773064  -55.7530077  -80.3058374  -24.84590252]
 [-17.14047155 -55.7530077  839.77520764 -41.15009043   2.20002553]
 [-24.84658715 -80.3058374  -41.15009043 760.56439687 -76.4962933 ]
 [-36.96701965 -24.84590252   2.20002553 -76.4962933  816.68257616]] 
 
 5 rows x 5 col of subject 1 
 [[ 669.51162683  -41.09744422   -5.80332711  -84.59222832  -53.40083341]
 [ -41.09744422  697.56114964  -71.47396287    8.26423805  -33.83418067]
 [  -5.80332711  -71.47396287  649.13342804 -102.70200543   28.29696551]
 [ -84.59222832    8.26423805 -102.70200543  763.21333262  -60.60025915]
 [ -53.40083341  -33.83418067   28.29696551  -60.60025915  813.09392363]] 
First 30 diagonal values of subject 0 
 [725.70683943 675.0773064  839.77520764 760.56439687 816.68257616
 818.2101299  726.74079318 702.65195073 713.53865459 859.09034248
 804.66901551 876.47646243 758.84764974 725.23552153 724

In [10]:
# def impute_diagonal_precision(
#         matrices: np.ndarray,
#         regularization: str = 'tikhonov',
#         alpha: float = 0.1,
#         normalize: bool = True,
# ) -> np.ndarray:
#     """
#     Impute diagonal using precision matrix (inverse covariance).

#     NEUROSCIENCE BASIS:
#     - Diagonal of precision matrix ≈ strength of anatomical self-connections
#     - Stronger in sensory/motor regions
#     - Captures direct dependencies (partial correlations)

#     Args:
#         matrices: (n_subjects, n_regions, n_regions) correlation matrices with diag = 1.0
#         region_list: Optional list of region names (unused here but kept for API consistency)
#         regularization: 'tikhonov' (recommended), 'none'
#         alpha: Regularization strength (Tikhonov); typical [0.01–0.5]
#         normalize: Scale precision diagonal to reasonable range

#     Returns:
#         matrices_imputed: Same shape, with diagonal replaced by precision diagonal
#     """
#     results = matrices.copy()
#     n_subjects, n_regions, _ = matrices.shape

#     for s in range(n_subjects):
#         corr = matrices[s].copy()

#         # Ensure valid correlation matrix 
#         np.fill_diagonal(corr, 1.0) 
#         corr = (corr + corr.T) / 2.0  # enforce perfect symmetry

#         # Apply regularization
#         if regularization == 'tikhonov':  # ridge regularization (L2)
#             regularized = corr + alpha * np.eye(n_regions)
#         elif regularization == 'none':
#             regularized = corr
#         else: 
#             raise ValueError(f"Invalid regularization: {regularization}")
        
#         # Invert to get precision matrix
#         precision = np.linalg.inv(regularized)
#         precision_diagonal = np.diag(precision)

#         # Normalise to avoid extreme values 
#         if normalize:
#             max_abs = np.max(np.abs(precision_diagonal)) 
            
#             # Scale to [-1,1] based on max_abs value
#             if max_abs > 1.0:
#                 precision_diagonal = precision_diagonal / max_abs
        
#         # Replace diagonal
#         np.fill_diagonal(results[s], precision_diagonal)

#     return results


# # CORRECT: Pass the entire 3D array at once
# df_mat_imputed = impute_diagonal_precision(df_mat, alpha=0.9)

# # Extract diagonal from first subject
# diagonal_values_1 = np.diagonal(df_mat_imputed[0])
# diagonal_values_2 = np.diagonal(df_mat_imputed[1])

# # First 30 diagonal values of subject 0
# print(diagonal_values_1[0:30])
# print('')
# print(diagonal_values_2[0:30])

# # first 5 
# print(f'\n 5 rows x 5 col of subject 0 \n {df_mat_imputed[0][0:5, 0:5]} ')


## 4. Preprocessing

### Fisher-z transformation

In [11]:
def fisher_z_transform(matrix, clip_value=0.999999):
    # Clip values to avoid infinities at r = ±1
    clipped_matrix = np.clip(matrix, -clip_value, clip_value)
    
    # Apply transformation to the CLIPPED matrix
    return np.arctanh(clipped_matrix)

# Apply fisher z transformation
df_mat_fz = np.array([fisher_z_transform(df_precision[i]) for i in range(df_precision.shape[0])])

# Extract diagonal from first subject
diagonal_values_1 = np.diagonal(df_mat_fz[0])
diagonal_values_2 = np.diagonal(df_mat_fz[1])

# First 30 diagonal values of subject 0
print(f'First 30 diagonal values of subject 0 \n {diagonal_values_1[0:30]}')
print('')
print(f'First 30 diagonal values of subject 1 \n {diagonal_values_1[0:30]}')


First 30 diagonal values of subject 0 
 [7.25432862 7.25432862 7.25432862 7.25432862 7.25432862 7.25432862
 7.25432862 7.25432862 7.25432862 7.25432862 7.25432862 7.25432862
 7.25432862 7.25432862 7.25432862 7.25432862 7.25432862 7.25432862
 7.25432862 7.25432862 7.25432862 7.25432862 7.25432862 7.25432862
 7.25432862 7.25432862 7.25432862 7.25432862 7.25432862 7.25432862]

First 30 diagonal values of subject 1 
 [7.25432862 7.25432862 7.25432862 7.25432862 7.25432862 7.25432862
 7.25432862 7.25432862 7.25432862 7.25432862 7.25432862 7.25432862
 7.25432862 7.25432862 7.25432862 7.25432862 7.25432862 7.25432862
 7.25432862 7.25432862 7.25432862 7.25432862 7.25432862 7.25432862
 7.25432862 7.25432862 7.25432862 7.25432862 7.25432862 7.25432862]


In [12]:
print(f'5 rows x 5 col of subject 0 \n {df_mat_fz[0][0:5, 0:5]} ')
print(f' \n 5 rows x 5 col of subject 1 \n {df_mat_fz[1][0:5, 0:5]} ')

5 rows x 5 col of subject 0 
 [[ 7.25432862 -7.25432862 -7.25432862 -7.25432862 -7.25432862]
 [-7.25432862  7.25432862 -7.25432862 -7.25432862 -7.25432862]
 [-7.25432862 -7.25432862  7.25432862 -7.25432862  7.25432862]
 [-7.25432862 -7.25432862 -7.25432862  7.25432862 -7.25432862]
 [-7.25432862 -7.25432862  7.25432862 -7.25432862  7.25432862]] 
 
 5 rows x 5 col of subject 1 
 [[ 7.25432862 -7.25432862 -7.25432862 -7.25432862 -7.25432862]
 [-7.25432862  7.25432862 -7.25432862  7.25432862 -7.25432862]
 [-7.25432862 -7.25432862  7.25432862 -7.25432862  7.25432862]
 [-7.25432862  7.25432862 -7.25432862  7.25432862 -7.25432862]
 [-7.25432862 -7.25432862  7.25432862 -7.25432862  7.25432862]] 


### Standard Scaler

In [13]:
from sklearn.preprocessing import StandardScaler

# Apply scaling to EACH subject's matrix (not the whole 3D array)
df_mat_fz_scaled = np.array([StandardScaler().fit_transform(df_mat_fz[i]) for i in range(df_mat_fz.shape[0])])

print(f'5 rows x 5 col of subject 0 \n {df_mat_fz_scaled[0][0:5, 0:5]} ')
print(f' \n 5 rows x 5 col of subject 1 \n {df_mat_fz_scaled[1][0:5, 0:5]} ')

5 rows x 5 col of subject 0 
 [[ 0.92339808 -0.94173312 -1.0425148  -1.03467859 -0.99977845]
 [-1.10182545  1.11425092 -1.0425148  -1.03467859 -0.99977845]
 [-1.10182545 -0.94173312  1.01063741 -1.03467859  1.03980431]
 [-1.10182545 -0.94173312 -1.0425148   1.00069676 -0.99977845]
 [-1.10182545 -0.94173312  1.01063741 -1.03467859  1.03980431]] 
 
 5 rows x 5 col of subject 1 
 [[ 0.91885252 -1.0670835  -0.99448178 -1.03632641 -0.89317104]
 [-1.14827885  0.96569877 -0.99448178  0.99474353 -0.89317104]
 [-1.14827885 -1.0670835   1.03654797 -1.03632641  1.18632471]
 [-1.14827885  0.96569877 -0.99448178  0.99474353 -0.89317104]
 [-1.14827885 -1.0670835   1.03654797 -1.03632641  1.18632471]] 


## 5. Modelling (StandardScaler)

In [14]:
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Prepare indices
np.random.seed(42)
X = df_mat_fz.reshape(-1, 232)
y = np.tile(np.arange(232), 224)
groups = np.repeat(np.arange(224), 232)

print(f"Shape of df_mat_fz: {df_precision.shape}")
print(f"Shape of X: {X.shape}")
print(f"Shape of y: {y.shape}")
print(f"Shape of groups: {groups.shape}")

gkf = GroupKFold(n_splits=3)
fold_scores = []

for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups)):

    # Split data
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    
    # Fit scaler on training data only, then transform both
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    
    # Train
    model = LogisticRegression(
        C=0.0343304473310619,
        max_iter=1000,
        solver='saga',
        multi_class='multinomial'
    )
    model.fit(X_train_scaled, y_train)
    
    # Evaluate
    score = accuracy_score(y_val, model.predict(X_val_scaled))

    # Store
    fold_scores.append(score)
    print(f"Fold {fold + 1} Accuracy: {score:.4f}")

print(f"\nMean CV Accuracy: {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}")


Shape of df_mat_fz: (224, 232, 232)
Shape of X: (51968, 232)
Shape of y: (51968,)
Shape of groups: (51968,)


/home/sjoon/projects/brain_connectivity_classifier/masterthesis_venv2/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Fold 1 Accuracy: 0.7756


/home/sjoon/projects/brain_connectivity_classifier/masterthesis_venv2/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Fold 2 Accuracy: 0.7561


/home/sjoon/projects/brain_connectivity_classifier/masterthesis_venv2/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Fold 3 Accuracy: 0.7582

Mean CV Accuracy: 0.7633 ± 0.0087
